# 2B core benchmark

Complete comparison of `no_loc`, integer-coordinate `loc_text` and L40 `loc_embed` full runs, including matched shuffled-coordinate controls.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

repo_root = Path.cwd()
if not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent

evaluation_root = repo_root / "outputs" / "evaluation"
runs = {
    "no_loc": "11437",
    "loc_text integer": "11441",
    "loc_embed L40": "11438",
}
shuffled_runs = {
    "loc_text integer": "11445",
    "loc_embed L40": "11446",
}
condition_order = list(runs)

def read_json(path):
    with path.open(encoding="utf-8") as handle:
        return json.load(handle)

summaries = {
    condition: read_json(evaluation_root / job / "scored_predictions" / "summary.json")
    for condition, job in runs.items()
}

predictions = {
    condition: pd.read_json(evaluation_root / job / "predictions.jsonl", lines=True)
    for condition, job in runs.items()
}

sample_scores = {
    condition: pd.read_json(
        evaluation_root / job / "scored_predictions" / "sample_scores.jsonl",
        lines=True,
    )
    for condition, job in runs.items()
}

pd.DataFrame({
    "Condition": condition_order,
    "Evaluation job": [runs[c] for c in condition_order],
    "Samples": [len(predictions[c]) for c in condition_order],
})

## Shuffled-coordinate controls

Values are shuffled minus correct. Negative values mean that replacing the true coordinates hurt performance.

In [ ]:
shuffled_summaries = {
    condition: read_json(evaluation_root / job / "scored_predictions" / "summary.json")
    for condition, job in shuffled_runs.items()
}

def counterfactual_task_row(summary, task_type):
    return next(row for row in summary["by_task_type"] if row["task_type"] == task_type)

def primary_metrics(summary):
    return {
        "Caption BLEU-4": summary["captioning"]["bleu4"],
        "Binary accuracy": counterfactual_task_row(summary, "binary")["accuracy"],
        "MCQ accuracy": counterfactual_task_row(summary, "mcq")["accuracy"],
        "Bounding-box mIoU": counterfactual_task_row(summary, "bounding box")["miou"],
    }

counterfactual_rows = []
for condition in shuffled_runs:
    correct = primary_metrics(summaries[condition])
    shuffled = primary_metrics(shuffled_summaries[condition])
    counterfactual_rows.append({
        "Condition": condition,
        **{metric: shuffled[metric] - correct[metric] for metric in correct},
    })

counterfactual_deltas = pd.DataFrame(counterfactual_rows).set_index("Condition")
counterfactual_deltas.style.format("{:+.4f}").background_gradient(cmap="RdYlGn", vmin=-counterfactual_deltas.abs().to_numpy().max(), vmax=counterfactual_deltas.abs().to_numpy().max()).set_caption("Shuffled − correct coordinates")

### Direct-geography MCQs under shuffling

In [ ]:
def category_accuracy(summary, category):
    return next(
        row["accuracy"]
        for row in summary["by_task_category"]
        if row["task_type"] == "mcq" and row["task_category"] == category
    )

geo_shuffle_rows = []
for condition in shuffled_runs:
    for category in ["country", "climate zone", "season"]:
        correct = category_accuracy(summaries[condition], category)
        shuffled = category_accuracy(shuffled_summaries[condition], category)
        geo_shuffle_rows.append({
            "Condition": condition,
            "Category": category,
            "Correct": correct,
            "Shuffled": shuffled,
            "Difference": shuffled - correct,
        })

pd.DataFrame(geo_shuffle_rows).style.format({"Correct": "{:.3f}", "Shuffled": "{:.3f}", "Difference": "{:+.3f}"})

## Population check

In [ ]:
id_sets = {condition: set(frame["sample_id"].astype(str)) for condition, frame in predictions.items()}
reference_ids = id_sets[condition_order[0]]

population_check = pd.DataFrame([
    {
        "Condition": condition,
        "Rows": len(predictions[condition]),
        "Unique sample IDs": predictions[condition]["sample_id"].astype(str).nunique(),
        "Same IDs as no_loc": ids == reference_ids,
    }
    for condition, ids in id_sets.items()
])
population_check

## Main results

One primary metric per task family. Average rank weights the four task families equally; lower is better.

In [ ]:
def task_row(summary, task_type):
    return next(row for row in summary["by_task_type"] if row["task_type"] == task_type)

main_results = pd.DataFrame([
    {
        "Condition": condition,
        "Caption BLEU-4": summary["captioning"]["bleu4"],
        "Binary accuracy": task_row(summary, "binary")["accuracy"],
        "MCQ accuracy": task_row(summary, "mcq")["accuracy"],
        "Bounding-box mIoU": task_row(summary, "bounding box")["miou"],
    }
    for condition, summary in summaries.items()
]).set_index("Condition").reindex(condition_order)

metric_columns = list(main_results.columns)
main_results["Average rank"] = main_results[metric_columns].rank(ascending=False).mean(axis=1)
main_results.style.format("{:.4f}").highlight_max(
    subset=metric_columns, axis=0, props="font-weight: bold"
).highlight_min(
    subset=["Average rank"], axis=0, props="font-weight: bold"
).set_caption("Primary benchmark metrics")

## Difference from no_loc

Positive values favor the location-conditioned model.

In [ ]:
delta = main_results.loc[["loc_text integer", "loc_embed L40"], metric_columns].subtract(main_results.loc["no_loc", metric_columns])
limit = delta.abs().to_numpy().max()
delta.style.format("{:+.4f}").background_gradient(cmap="RdYlGn", vmin=-limit, vmax=limit).set_caption("Location condition − no_loc")

## Task-wise results

In [ ]:
category_rows = []
for condition, summary in summaries.items():
    for row in summary["by_task_category"]:
        category_rows.append({"Condition": condition, **row})
category_scores = pd.DataFrame(category_rows)

def category_table(task_type, metric):
    rows = category_scores[category_scores["task_type"] == task_type]
    table = rows.pivot(index="Condition", columns="task_category", values=metric)
    overall = pd.Series({
        condition: task_row(summaries[condition], task_type)[metric]
        for condition in condition_order
    }, name="Overall")
    return table.reindex(condition_order).join(overall)

display(category_table("binary", "accuracy").style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold").set_caption("Binary accuracy"))
display(category_table("mcq", "accuracy").style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold").set_caption("MCQ accuracy"))
display(category_table("bounding box", "miou").style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold").set_caption("Bounding-box mIoU"))

## Geography-sensitive MCQs

Country, climate-zone and season questions are the clearest direct test of whether the location tokens carry useful geographic information.

In [ ]:
geo_categories = ["country", "climate zone", "season"]
geo_mcq = (
    category_scores[
        (category_scores["task_type"] == "mcq")
        & category_scores["task_category"].isin(geo_categories)
    ]
    .pivot(index="Condition", columns="task_category", values="accuracy")
    .reindex(condition_order)
)
geo_mcq["Mean"] = geo_mcq.mean(axis=1)
geo_mcq.style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold").set_caption("Direct-geography MCQ accuracy")

## Paired changes

Counts below show whether `loc_embed` fixes or breaks the exact same binary and MCQ samples.

In [ ]:
base = sample_scores["no_loc"][["sample_id", "task_type", "task_category", "correct"]].rename(columns={"correct": "no_loc_correct"})
embed = sample_scores["loc_embed L40"][["sample_id", "correct"]].rename(columns={"correct": "loc_embed_correct"})
paired = base.merge(embed, on="sample_id", validate="one_to_one")
paired = paired[paired["task_type"].isin(["binary", "mcq"])].copy()

def transition(row):
    if row.no_loc_correct and not row.loc_embed_correct:
        return "Broken by loc_embed"
    if not row.no_loc_correct and row.loc_embed_correct:
        return "Fixed by loc_embed"
    if row.no_loc_correct:
        return "Both correct"
    return "Both wrong"

paired["Transition"] = paired.apply(transition, axis=1)
transition_table = pd.crosstab(
    [paired["task_type"], paired["task_category"]],
    paired["Transition"],
).fillna(0).astype(int)
transition_table

## Qualitative changes

Examples where the two models produced different answers. Change `task_filter` and `category_filter` to inspect a weakness or a location-sensitive subtask.

In [ ]:
task_filter = "mcq"
category_filter = "country"
number_of_examples = 12

keep = ["sample_id", "input_text", "target_texts", "prediction", "task_type", "task_category", "country", "lat", "lon"]
base_predictions = predictions["no_loc"][keep].rename(columns={"prediction": "no_loc prediction"})
embed_predictions = predictions["loc_embed L40"][["sample_id", "prediction"]].rename(columns={"prediction": "loc_embed prediction"})
comparison = base_predictions.merge(embed_predictions, on="sample_id", validate="one_to_one")
changed = comparison[
    (comparison["task_type"] == task_filter)
    & (comparison["task_category"] == category_filter)
    & (comparison["no_loc prediction"] != comparison["loc_embed prediction"])
]
changed.head(number_of_examples)

## Full diagnostic tables

These retain sample counts, extraction rates, secondary caption metrics and bounding-box thresholds.

In [ ]:
task_type_rows = []
caption_rows = []
for condition, summary in summaries.items():
    for row in summary["by_task_type"]:
        task_type_rows.append({"Condition": condition, **row})
    caption_rows.append({"Condition": condition, **summary["captioning"]})

display(pd.DataFrame(task_type_rows).sort_values(["task_type", "Condition"]).reset_index(drop=True))
display(pd.DataFrame(caption_rows).set_index("Condition").reindex(condition_order).reset_index())
display(category_scores.sort_values(["task_type", "task_category", "Condition"]).reset_index(drop=True))

## Current reading

- `loc_embed` has the best MCQ accuracy; its gain is concentrated in direct-geography questions.
- `loc_text` has the best CIDEr, while `loc_embed` has the best BLEU-4, so captioning has no single winner across metrics.
- `no_loc` has the best binary accuracy and bounding-box mIoU.
- Shuffling strongly damages captioning and MCQ, confirming that both location-conditioned models use their coordinates.